# TweetAge v7: descuento con concentración

**interpolación descontada con parámetro de concentración, basada en Chen y Goodman (1999)**. Se mantienen los conteos, D1/D2/D3+ y la base de continuaciones de v6. Solo cambia la interpolacion al incorporar theta. Teh (2006) se cita unicamente como inspiracion para explorar concentracion; esta implementacion no es Pitman-Yor exacto ni inferencia bayesiana y no aprende variables de mesas.

Para cada historial u, con R(u) igual a la suma de los descuentos retirados:

$$P(w|u)=\frac{\max(c(uw)-D(c(uw)),0)+(\theta+R(u))P(w|u')}{\theta+c(u)}.$$

La suma del numerador es c(u)-R(u)+theta+R(u)=c(u)+theta si la distribucion inferior suma uno. Se prueban ambos theta={0.5,1.0}, todos los historiales y ordenes del corpus sintetico, en caracteres y palabras. En los modelos reales se verifica la suma completa mediante agregacion de masa por historial y recursividad, sin recorrer un vocabulario entero por cada historial; no es muestreo. Los historiales no observados retroceden a la distribucion inferior, normalizada.

Caracteres orden 4, palabras orden 3, lambda=0.75, score bruto, priors de ajuste, sin sesgos calibrados ni clasificadores sobre scores. Se elige solo theta en cinco pliegues agrupados del desarrollo de v6, semilla 42. Si el mejor F1 macro OOF sin calibrar no alcanza 0.44306, se omiten holdout y CSV.

V5 tenia fuga por textos repetidos; v6 separa grupos. Se reutiliza el NPZ, no se regenera. La seleccion de theta tambien introduce optimismo de seleccion OOF, aunque no haya calibracion. Los scores de Kaggle no son comparables directamente con holdout.

Referencias: Chen, S. F. y Goodman, J. (1999), An empirical study of smoothing techniques for language modeling, Computer Speech & Language, 13(4), 359-394. Teh, Y. W. (2006), A hierarchical Bayesian language model based on Pitman-Yor processes, COLING/ACL: solo inspiracion, no descripcion del algoritmo implementado. La consulta bibliografica no implica validacion experimental de esta variante.

In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
CLASSES = np.array(['13-17','18-24','25-34','35-49','50-64','65-xx'])
CHAR_ORDER,WORD_ORDER,WEIGHT = 4,3,0.75
THETAS = [0.5,1.0]
GATE_TARGET = 0.44306
BASE_EPSILON = 1e-6
BOS,EOS,UNK = '\x02','\x03','\x01'
roots = [Path.cwd(),Path.cwd()/'workshops'/'Kaggle-tweet-age',Path.cwd()/'natural-language-processing-workshops'/'workshops'/'Kaggle-tweet-age']
OUT_DIR = next((candidate for candidate in roots if (candidate/'TweetAge_v7_descuento_concentracion.ipynb').is_file()),None)
assert OUT_DIR is not None,'Ejecutar desde la carpeta del notebook o la raiz del repo.'
NOTEBOOK_PATH = OUT_DIR/'TweetAge_v7_descuento_concentracion.ipynb'
SUB_PATH = OUT_DIR/'age_submission_v7_descuento_concentracion.csv'
DATA_DIR = OUT_DIR/'tweet-age-2026-puj'
SPLIT_PATH = OUT_DIR/'split_idx_v6_seed42.npz'
assert not SUB_PATH.exists(),'El CSV ya existe; no se sobrescribe.'
INPUT_HASHES = {name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in ['age_train.csv','age_test.csv','age_submission.csv']}
PROTECTED_HASHES = {name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in ['TweetAge_v5_charLM_kn.ipynb','TweetAge_v6_charLM_kn.ipynb','TweetAge_v7_ordinal_kn.ipynb','split_idx_v6_seed42.npz','requirements.txt']}
print('Python',platform.python_version(),'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__)
print('Semilla',SEED,'notebook',NOTEBOOK_PATH.name,'CSV condicional',SUB_PATH.name)
print('SHA-256 age_train.csv:',INPUT_HASHES['age_train.csv'])
def check_compliance(notebook):
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST de v6 superada antes de entrenar.')


Python 3.12.14 numpy 2.5.4 pandas 3.0.6 sklearn 1.9.1
Semilla 42 notebook TweetAge_v7_descuento_concentracion.ipynb CSV condicional age_submission_v7_descuento_concentracion.csv
SHA-256 age_train.csv: 39c6c8270366fe155f3c8dd232e31867bee659af8908121d04b6bd89bd163748
Guardia AST de v6 superada antes de entrenar.


In [2]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

verify_csv(DATA_DIR / 'age_train.csv', ['id','text','age_range'], ';', True)
reference_columns = verify_csv(DATA_DIR / 'age_submission.csv', ['id','age_range'], ',', False)
train_raw = pd.read_csv(DATA_DIR / 'age_train.csv', sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert list(train_raw.columns) == ['id','text','age_range']
clean_text = train_raw.text.fillna('').str.strip().map(html.unescape)
excluded = clean_text.eq('') | clean_text.str.contains('#\u00bfNOMBRE', regex=False)
train = train_raw.loc[~excluded].copy()
train['raw_row'] = np.flatnonzero(~excluded.to_numpy())
train['text'] = clean_text.loc[~excluded]
train = train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range) == set(CLASSES)
train['normalized'] = train.text.map(normalize_text)
y_all = pd.Categorical(train.age_range, categories=CLASSES).codes.astype(np.int64)
with np.load(SPLIT_PATH, allow_pickle=False) as saved:
    split = {name: saved[name].copy() for name in saved.files}
assert str(split['train_sha256']) == INPUT_HASHES['age_train.csv'], 'SHA-256 del train distinto al NPZ; detener.'
assert int(split['seed']) == SEED
assert np.array_equal(split['retained_raw_idx'], train.raw_row.to_numpy())
idx_development, idx_holdout = split['train_idx'], split['valid_idx']
assert np.array_equal(np.sort(np.concatenate([idx_development,idx_holdout])), np.arange(len(train)))
assert not np.intersect1d(idx_development,idx_holdout).size
actual_groups, unique_texts = pd.factorize(train.normalized, sort=True)
assert np.array_equal(actual_groups,split['group_ids']), 'Normalizacion o grupos distintos de v6.'
assert np.array_equal(train.raw_row.to_numpy()[idx_development],split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout],split['raw_valid_idx'])
dev_groups, holdout_groups = split['train_groups'],split['valid_groups']
assert set(actual_groups[idx_development]) == set(dev_groups)
assert set(actual_groups[idx_holdout]) == set(holdout_groups)
intersection = set(train.normalized.iloc[idx_development]) & set(train.normalized.iloc[idx_holdout])
assert not intersection
texts_development = train.normalized.iloc[idx_development].to_numpy()
y_development = y_all[idx_development]
groups_development = actual_groups[idx_development]
group_counts = np.zeros((len(unique_texts),6),dtype=np.int64)
np.add.at(group_counts,(actual_groups,y_all),1)
group_targets = group_counts.argmax(axis=1)
print('NPZ reutilizado, sin regenerar:', SPLIT_PATH.name, '| SHA-256 correcto')
print('Train original/limpio/excluidos:', len(train_raw),len(train),int(excluded.sum()))
print('Desarrollo grupos/filas:',len(dev_groups),len(idx_development),'| Holdout grupos/filas:',len(holdout_groups),len(idx_holdout))
print('Interseccion de textos normalizados:',len(intersection),'| Solo URL train:',int(train.normalized.str.strip().eq('<url>').sum()))
display(train.age_range.value_counts().reindex(CLASSES).rename('filas').to_frame())


age_train.csv BOM= True separador= ';' columnas= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' columnas= ['id', 'age_range']


NPZ reutilizado, sin regenerar: split_idx_v6_seed42.npz | SHA-256 correcto
Train original/limpio/excluidos: 169336 169181 155
Desarrollo grupos/filas: 126232 133065 | Holdout grupos/filas: 31558 36116
Interseccion de textos normalizados: 0 | Solo URL train: 3785


,filas
age_range,
13-17,28785
18-24,28626
25-34,28324
35-49,28226
50-64,28060
65-xx,27160


In [3]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

class DiscountConcentrationLM(ModifiedKN):
    def __init__(self,max_order,support,theta=0.5):
        super().__init__(max_order,support)
        assert theta > 0
        self.theta = float(theta)

    def set_theta(self,theta):
        assert theta > 0
        self.theta = float(theta)
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

    def _lower(self,gram):
        if len(gram)==1:
            return (self.base.get(gram,0)+BASE_EPSILON)/self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total,removed = self.cont_hist[order].get(gram[:-1],(0,0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram,0)
        numerator = max(count-discount_for(count,self.cont_discounts[order]),0)
        return (numerator+(self.theta+removed)*lower)/(self.theta+total)

    def probability(self,gram):
        if len(gram)==1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total,removed = self.raw_hist[order].get(gram[:-1],(0,0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram,0)
        numerator = max(count-discount_for(count,self.raw_discounts[order]),0)
        return (numerator+(self.theta+removed)*lower)/(self.theta+total)


def normalization_mass_check(model,theta):
    model.set_theta(theta)
    base_mass = (math.fsum(model.base.values())+BASE_EPSILON*len(model.support))/model.base_total
    max_error = abs(base_mass-1.0)
    checked = 1
    lower_masses = {'':base_mass}
    for order in sorted(model.cont_hist):
        observed = Counter()
        for gram,count in model.continuation[order].items():
            observed[gram[:-1]] += max(count-discount_for(count,model.cont_discounts[order]),0)
        current_masses = {}
        for history,(total,removed) in model.cont_hist[order].items():
            lower = lower_masses.get(history[1:],base_mass)
            mass = (observed[history]+(theta+removed)*lower)/(theta+total)
            current_masses[history] = mass
            max_error = max(max_error,abs(mass-1.0))
            checked += 1
        for history,(total,removed) in model.raw_hist[order].items():
            pass
        lower_masses = current_masses
    lower_by_order = {1:{'':base_mass}}
    for order in sorted(model.cont_hist):
        observed = Counter()
        for gram,count in model.continuation[order].items():
            observed[gram[:-1]] += max(count-discount_for(count,model.cont_discounts[order]),0)
        previous = lower_by_order[order-1]
        lower_by_order[order] = {history:(observed[history]+(theta+removed)*previous.get(history[1:],base_mass))/(theta+total) for history,(total,removed) in model.cont_hist[order].items()}
    for order in sorted(model.raw_hist):
        observed = Counter()
        for gram,count in model.raw[order].items():
            observed[gram[:-1]] += max(count-discount_for(count,model.raw_discounts[order]),0)
        previous = lower_by_order[order-1]
        for history,(total,removed) in model.raw_hist[order].items():
            mass = (observed[history]+(theta+removed)*previous.get(history[1:],base_mass))/(theta+total)
            max_error = max(max_error,abs(mass-1.0))
            checked += 1
    if max_error > 1e-9:
        raise ArithmeticError(('Falla normalizacion',theta,checked,max_error))
    return checked,max_error


def normalization_preflight():
    records = []
    toy_texts = ['Hola hola!','Hola amiga!','Adios hola!','Adios amiga!','Hola hola!']
    for level,max_order in [('char',CHAR_ORDER),('word',WORD_ORDER)]:
        mapping = make_encoder(toy_texts,level)
        sequences = [encode_text(text,level,mapping) for text in toy_texts]
        model = DiscountConcentrationLM(max_order,set(mapping.values())).fit(sequences)
        for theta in THETAS:
            model.set_theta(theta)
            total_checked,max_error = normalization_mass_check(model,theta)
            checks = 0
            for origin,tables,histories in [('frecuencias',model.raw,model.raw_hist),('continuaciones',model.continuation,model.cont_hist)]:
                for order in sorted(histories):
                    for history in set(histories[order]) | {'Z'*(order-1)}:
                        if origin=='frecuencias':
                            mass = math.fsum(model.probability(history+token) for token in model.support)
                        else:
                            mass = math.fsum(model.lower_cached(history+token) for token in model.support)
                        max_error = max(max_error,abs(mass-1.0))
                        checks += 1
                        assert abs(mass-1.0)<=1e-9,(level,theta,origin,order,repr(history),mass)
            assert abs(math.fsum(model.probability(token) for token in model.support)-1)<=1e-9
            assert math.isfinite(model.log_probability(UNK,max_order))
            records.append(dict(nivel=level,theta=theta,historiales_agregados=total_checked,historiales_suma_directa=checks,error_maximo=max_error))
        model.close()
    return pd.DataFrame(records)

preflight_table = normalization_preflight()
print('PRUEBA PREVIA NORMALIZACION: PASA para todos los historiales, ordenes, niveles y ambos theta; tolerancia 1e-9.')
display(preflight_table)
normalization_rows,timing_rows = [],[]

def fit_score_level(fit_texts,fit_labels,app_texts,level,order,thetas,phase,fold):
    started = time.perf_counter()
    mapping = make_encoder(fit_texts,level)
    encoded_fit = [encode_text(text,level,mapping) for text in fit_texts]
    encoded_app = [encode_text(text,level,mapping) for text in app_texts]
    preprocessing = time.perf_counter()-started
    matrices = {theta:np.zeros((len(app_texts),6)) for theta in thetas}
    shared_fit,scoring = 0.0,{theta:0.0 for theta in thetas}
    for class_index,label in enumerate(CLASSES):
        fit_started = time.perf_counter()
        model = DiscountConcentrationLM(order,set(mapping.values())).fit(sequence for sequence,target in zip(encoded_fit,fit_labels) if target==class_index)
        shared_fit += time.perf_counter()-fit_started
        for theta in THETAS:
            checks,error = normalization_mass_check(model,theta)
            normalization_rows.append(dict(fase=phase,pliegue=fold,nivel=level,clase=str(label),theta=theta,historiales=checks,error_maximo=error))
        for theta in thetas:
            score_started = time.perf_counter()
            model.set_theta(theta)
            matrices[theta][:,class_index] = [model.log_probability(sequence,order) for sequence in encoded_app]
            scoring[theta] += time.perf_counter()-score_started
        model.close()
        del model
        gc.collect()
        print(phase,fold,level,str(label),'listo',round(time.perf_counter()-started,2),'s',flush=True)
    for theta in thetas:
        timing_rows.append(dict(fase=phase,pliegue=fold,nivel=level,theta=theta,ajuste_compartido_s=shared_fit+preprocessing,puntuacion_s=scoring[theta],tiempo_total_nivel_s=time.perf_counter()-started))
    return matrices


PRUEBA PREVIA NORMALIZACION: PASA para todos los historiales, ordenes, niveles y ambos theta; tolerancia 1e-9.


,nivel,theta,historiales_agregados,historiales_suma_directa,error_maximo
0,char,0.5,93,97,2.220446e-16
1,char,1.0,93,97,2.220446e-16
2,word,0.5,20,22,2.220446e-16
3,word,1.0,20,22,2.220446e-16


In [4]:
pilot_groups,_ = train_test_split(dev_groups,train_size=0.1,stratify=group_targets[dev_groups],random_state=SEED)
pilot_fit_groups,pilot_app_groups = train_test_split(pilot_groups,test_size=0.2,stratify=group_targets[pilot_groups],random_state=SEED)
pilot_fit = np.flatnonzero(np.isin(groups_development,pilot_fit_groups))
pilot_app = np.flatnonzero(np.isin(groups_development,pilot_app_groups))
assert not set(texts_development[pilot_fit]) & set(texts_development[pilot_app])
pilot_started = time.perf_counter()
pilot_char = fit_score_level(texts_development[pilot_fit],y_development[pilot_fit],texts_development[pilot_app],'char',CHAR_ORDER,THETAS,'piloto_10pct',0)
pilot_word = fit_score_level(texts_development[pilot_fit],y_development[pilot_fit],texts_development[pilot_app],'word',WORD_ORDER,THETAS,'piloto_10pct',0)
pilot_prior = np.log(np.bincount(y_development[pilot_fit],minlength=6)/len(pilot_fit))
for theta in THETAS:
    predictions = (WEIGHT*pilot_char[theta]+(1-WEIGHT)*pilot_word[theta]+pilot_prior).argmax(axis=1)
    print('Piloto theta',theta,'F1 macro',f1_score(y_development[pilot_app],predictions,labels=np.arange(6),average='macro',zero_division=0))
print('Piloto segundos:',round(time.perf_counter()-pilot_started,2),'ambos theta se mantienen fijados; no se decide con el piloto.')
display(pd.DataFrame(timing_rows))
del pilot_char,pilot_word
gc.collect()


piloto_10pct 0 char 13-17 listo 0.29 s


piloto_10pct 0 char 18-24 listo 0.52 s


piloto_10pct 0 char 25-34 listo 0.76 s


piloto_10pct 0 char 35-49 listo 1.0 s


piloto_10pct 0 char 50-64 listo 1.25 s


...

piloto_10pct 0 word 50-64 listo 0.77 s


piloto_10pct 0 word 65-xx listo 0.92 s


Piloto theta 0.5 F1 macro 0.3461802908663694
Piloto theta 1.0 F1 macro 0.3467583865323556
Piloto segundos: 2.42 ambos theta se mantienen fijados; no se decide con el piloto.


,fase,pliegue,nivel,theta,ajuste_compartido_s,puntuacion_s,tiempo_total_nivel_s
0,piloto_10pct,0,char,0.5,0.372230,0.395861,1.494705
1,piloto_10pct,0,char,1.0,0.372230,0.396322,1.494708
2,piloto_10pct,0,word,0.5,0.211089,0.187066,0.922878
3,piloto_10pct,0,word,1.0,0.211089,0.187854,0.922880


0

In [5]:
cv_started = time.perf_counter()
group_folds = list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
oof_scores = {theta:np.zeros((len(y_development),6)) for theta in THETAS}
fold_id = np.full(len(y_development),-1,dtype=int)
folds = []
for fold_number,(fit_group_positions,app_group_positions) in enumerate(group_folds,1):
    fit_idx = np.flatnonzero(np.isin(groups_development,dev_groups[fit_group_positions]))
    app_idx = np.flatnonzero(np.isin(groups_development,dev_groups[app_group_positions]))
    assert not set(texts_development[fit_idx]) & set(texts_development[app_idx])
    folds.append((fit_idx,app_idx))
    characters = fit_score_level(texts_development[fit_idx],y_development[fit_idx],texts_development[app_idx],'char',CHAR_ORDER,THETAS,'CV',fold_number)
    words = fit_score_level(texts_development[fit_idx],y_development[fit_idx],texts_development[app_idx],'word',WORD_ORDER,THETAS,'CV',fold_number)
    prior = np.log(np.bincount(y_development[fit_idx],minlength=6)/len(fit_idx))
    for theta in THETAS:
        oof_scores[theta][app_idx] = WEIGHT*characters[theta]+(1-WEIGHT)*words[theta]+prior
    fold_id[app_idx] = fold_number
    del characters,words
    gc.collect()
    print('Pliegue',fold_number,'completo; interseccion normalizada 0',flush=True)
assert (fold_id>0).all()
cv_rows = []
for theta in THETAS:
    pred = oof_scores[theta].argmax(axis=1)
    fold_metrics = [f1_score(y_development[app_idx],pred[app_idx],labels=np.arange(6),average='macro',zero_division=0) for _,app_idx in folds]
    cv_rows.append(dict(theta=theta,f1_macro_oof=f1_score(y_development,pred,labels=np.arange(6),average='macro',zero_division=0),media_pliegues=float(np.mean(fold_metrics)),std_pliegues=float(np.std(fold_metrics))))
cv_results = pd.DataFrame(cv_rows).sort_values(['f1_macro_oof','theta'],ascending=[False,True]).reset_index(drop=True)
best_theta = float(cv_results.iloc[0].theta)
best_oof = float(cv_results.iloc[0].f1_macro_oof)
oof_pred = oof_scores[best_theta].argmax(axis=1)
print('TABLA OOF SIN CALIBRACION')
print(cv_results.to_string(index=False))
print('Theta elegido',best_theta,'| CV segundos',round(time.perf_counter()-cv_started,2))
normalization_table = pd.DataFrame(normalization_rows)
assert normalization_table.error_maximo.max()<=1e-9
print('Normalizacion real por todos los historiales, todos los niveles y ambos theta: PASA; error maximo',normalization_table.error_maximo.max())
display(normalization_table.groupby(['nivel','theta']).agg(modelos=('clase','size'),historiales=('historiales','sum'),error_maximo=('error_maximo','max')))
display(pd.DataFrame(timing_rows))


CV 1 char 13-17 listo 2.08 s


CV 1 char 18-24 listo 4.11 s


CV 1 char 25-34 listo 5.93 s


CV 1 char 35-49 listo 7.67 s


CV 1 char 50-64 listo 9.37 s


...

CV 5 word 65-xx listo 11.79 s


Pliegue 5 completo; interseccion normalizada 0


TABLA OOF SIN CALIBRACION
 theta  f1_macro_oof  media_pliegues  std_pliegues
   0.5      0.440875        0.440854      0.003664
   1.0      0.440788        0.440764      0.003455
Theta elegido 0.5 | CV segundos 109.03
Normalizacion real por todos los historiales, todos los niveles y ambos theta: PASA; error maximo 1.2656542480726785e-13


modelos  historiales  error_maximo
nivel theta                                    
char  0.5         36      1214950  6.106227e-15
      1.0         36      1214950  6.106227e-15
word  0.5         36      4794477  1.265654e-13
      1.0         36      4794477  1.265654e-13

,fase,pliegue,nivel,theta,ajuste_compartido_s,puntuacion_s,tiempo_total_nivel_s
0,piloto_10pct,0,char,0.5,0.372230,0.395861,1.494705
1,piloto_10pct,0,char,1.0,0.372230,0.396322,1.494708
2,piloto_10pct,0,word,0.5,0.211089,0.187066,0.922878
3,piloto_10pct,0,word,1.0,0.211089,0.187854,0.922880
4,CV,1,char,0.5,3.187058,3.445684,11.084295
5,CV,1,char,1.0,3.187058,3.636995,11.084296
6,CV,1,word,0.5,2.263329,3.075062,10.794189
7,CV,1,word,1.0,2.263329,3.199565,10.794190
8,CV,2,char,0.5,3.038448,3.248093,10.403028
9,CV,2,char,1.0,3.038448,3.325881,10.403029


In [6]:
PASA = bool(best_oof>=GATE_TARGET)
print('PASA' if PASA else 'NO PASA')
print('Mejor F1 OOF sin calibrar',best_oof,'| objetivo',GATE_TARGET,'| delta frente a v6 sin calibrar',best_oof-0.44106)
print('Sin calibracion de sesgos; solo se eligio theta en CV agrupada de desarrollo.')
if not PASA:
    print('NO PASA: holdout y prediccion final se omiten; no se creara age_submission_v7_descuento_concentracion.csv.')


NO PASA
Mejor F1 OOF sin calibrar 0.44087504395036453 | objetivo 0.44306 | delta frente a v6 sin calibrar -0.0001849560496354763
Sin calibracion de sesgos; solo se eligio theta en CV agrupada de desarrollo.
NO PASA: holdout y prediccion final se omiten; no se creara age_submission_v7_descuento_concentracion.csv.


In [7]:
if PASA:
    assert 'holdout_f1' not in globals(),'Holdout ya evaluado; no reutilizar.'
    holdout_started = time.perf_counter()
    texts_holdout = train.normalized.iloc[idx_holdout].to_numpy()
    characters = fit_score_level(texts_development,y_development,texts_holdout,'char',CHAR_ORDER,[best_theta],'holdout',0)
    words = fit_score_level(texts_development,y_development,texts_holdout,'word',WORD_ORDER,[best_theta],'holdout',0)
    prior = np.log(np.bincount(y_development,minlength=6)/len(y_development))
    holdout_pred = (WEIGHT*characters[best_theta]+(1-WEIGHT)*words[best_theta]+prior).argmax(axis=1)
    y_holdout = y_all[idx_holdout]
    holdout_f1 = f1_score(y_holdout,holdout_pred,labels=np.arange(6),average='macro',zero_division=0)
    print('F1 macro holdout:',holdout_f1)
    report = classification_report(y_holdout,holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)
    display(pd.DataFrame(report).T.loc[CLASSES,['precision','recall','f1-score','support']])
    display(pd.DataFrame(confusion_matrix(y_holdout,holdout_pred,labels=np.arange(6)),index=CLASSES,columns=CLASSES))
    print('Delta descriptivo v5:',holdout_f1-0.44945,'v6:',holdout_f1-0.41847,'; v5 usa otro split.')
    print('Holdout segundos',round(time.perf_counter()-holdout_started,2))
    del characters,words
    gc.collect()
else:
    print('HOLDOUT OMITIDO: NO PASA; no se entrena ni se evalua sobre holdout.')


HOLDOUT OMITIDO: NO PASA; no se entrena ni se evalua sobre holdout.


In [8]:
if PASA:
    verify_csv(DATA_DIR/'age_test.csv',['id','text'],';',True)
    test = pd.read_csv(DATA_DIR/'age_test.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    assert list(test.columns)==['id','text'] and test.id.is_unique and test.id.ne('').all()
    test_texts = test.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy()
    characters = fit_score_level(train.normalized.to_numpy(),y_all,test_texts,'char',CHAR_ORDER,[best_theta],'final',0)
    words = fit_score_level(train.normalized.to_numpy(),y_all,test_texts,'word',WORD_ORDER,[best_theta],'final',0)
    prior = np.log(np.bincount(y_all,minlength=6)/len(y_all))
    predictions = (WEIGHT*characters[best_theta]+(1-WEIGHT)*words[best_theta]+prior).argmax(axis=1)
    submission = pd.DataFrame({'id':test.id,'age_range':CLASSES[predictions]})
    assert list(submission.columns)==reference_columns==['id','age_range']
    assert len(submission)==len(test)==112892 and submission.id.is_unique
    assert submission.id.tolist()==test.id.tolist() and submission.age_range.isin(CLASSES).all()
    with SUB_PATH.open('x',encoding='utf-8',newline='') as handle:
        submission.to_csv(handle,index=False,sep=',')
    with SUB_PATH.open('rb') as handle: prefix=handle.read(64)
    assert not prefix.startswith(b'\xef\xbb\xbf') and prefix.splitlines()[0]==b'id,age_range'
    print('CSV VERIFICADO:',SUB_PATH.name,'112892 filas; IDs iguales y en orden; clases validas; coma y sin BOM.')
    del characters,words
    gc.collect()
else:
    assert not SUB_PATH.exists()
    print('CSV OMITIDO: NO PASA; no se carga age_test.csv para predecir ni se crea salida.')


CSV OMITIDO: NO PASA; no se carga age_test.csv para predecir ni se crea salida.


In [9]:
if PASA:
    error_targets,error_predictions,error_indices,error_texts = y_holdout,holdout_pred,idx_holdout,texts_holdout
    error_origin = 'holdout'
else:
    error_targets,error_predictions,error_indices,error_texts = y_development,oof_pred,idx_development,texts_development
    error_origin = 'OOF desarrollo'

def interpret_error(text,real,predicted):
    if text.strip()=='<url>': return 'Hipotesis: una URL aislada no aporta contenido lexico de edad.'
    if len(TOKEN_RE.findall(text))<5: return 'Hipotesis: texto muy corto para distinguir rangos.'
    if abs(real-predicted)==1: return 'Hipotesis: rangos contiguos comparten vocabulario y estilo.'
    if '<user>' in text: return 'Hipotesis: normalizar cuentas elimina una pista de comunidad.'
    return 'Hipotesis: contenido compartido entre edades; requiere revision humana.'

positions = np.flatnonzero(error_targets!=error_predictions)[:10]
error_table = pd.DataFrame([dict(texto=train.text.iloc[error_indices[position]],real=CLASSES[error_targets[position]],predicha=CLASSES[error_predictions[position]],interpretacion=interpret_error(error_texts[position],int(error_targets[position]),int(error_predictions[position]))) for position in positions])
print('Diez errores de',error_origin,'; interpretaciones marcadas como hipotesis.')
with pd.option_context('display.max_colwidth',180): display(error_table)


Diez errores de OOF desarrollo ; interpretaciones marcadas como hipotesis.


,texto,real,predicha,interpretacion
0,RT @NintendoAmerica: Fans will get a chance to play the latest Fatal Frame game when it launches this year for #WiiU. #NintendoDirectNA htt,13-17,25-34,Hipotesis: normalizar cuentas elimina una pista de comunidad.
1,@Marscia_02 vos tb!,35-49,25-34,Hipotesis: texto muy corto para distinguir rangos.
2,"Corte Constitucional, SC-035: El agua es insustituible para la salud y la vida humana, patrimonio de la nacion #DiaMundialDelAgua",35-49,50-64,Hipotesis: rangos contiguos comparten vocabulario y estilo.
3,RT @JaimeNio: @miguelale @USAenEspanol @SuperSONICAX @statedeptspox AHI SE LO ENVIO! SIN SOBRE VOLAR ESPACIO AEREO! https://t.co/kmut3n24HT,65-xx,50-64,Hipotesis: rangos contiguos comparten vocabulario y estilo.
4,@Fernand41939055 como vamos fercho con esos negocios,25-34,50-64,Hipotesis: normalizar cuentas elimina una pista de comunidad.
5,"RT @Arenita_blue: No saben de alguien que quiera un Golden? Que tenga espacio, disposicion y tiempo de cuidarlo bien https://t.co/GmcW",25-34,35-49,Hipotesis: rangos contiguos comparten vocabulario y estilo.
6,RT @nohaynadie_: Mientras tanto Paco Moncayo https://t.co/bvIbY9RFJw,35-49,50-64,Hipotesis: rangos contiguos comparten vocabulario y estilo.
7,@CamiloFR10 ahora que no lo vayan a dejar la verdad llego a la final por un buen partido con medellin pero mal todo el ano,50-64,18-24,Hipotesis: normalizar cuentas elimina una pista de comunidad.
8,RT @elmonitosanchez: Santofimio casa por carcel....cuando llegara DMG? Cuando liberaran al que se robo un cubo de caldo? Cumplira condena e,65-xx,50-64,Hipotesis: rangos contiguos comparten vocabulario y estilo.
9,RT @BA_Cultura: #AcercArte te cuenta #Gratis Que leen los actores? en @LaPlataMLP y con @gonzaloherediao https://t.co/kF158IJNom,18-24,25-34,Hipotesis: rangos contiguos comparten vocabulario y estilo.


In [10]:
summary_table = pd.DataFrame({'version':['v5 KN historica','v6 KN agrupada','v7 KN por cortes','v7 descuento-concentracion'],'OOF_sin_calibrar':[0.43539,0.44106,0.42321,best_oof],'OOF_calibrado':[0.43847,0.44353,0.42468,np.nan],'holdout':[0.44945,0.41847,np.nan,holdout_f1 if PASA else np.nan]})
display(summary_table)
summary_text = f"""## Resumen final

Modelo: **interpolación descontada con parámetro de concentración, basada en Chen y Goodman (1999)**. Teh (2006) es solo inspiracion; no es Pitman-Yor exacto ni inferencia bayesiana.

Prueba de normalizacion: **PASA**, tolerancia **1e-9**, ambos theta, todos los historiales/ordenes en caracteres y palabras, con suma directa en el corpus sintetico y agregacion exacta de masa en los modelos reales. Error maximo real: **{pd.DataFrame(normalization_rows).error_maximo.max():.3e}**.

Theta elegido: **{best_theta}**; F1 OOF sin calibrar: **{best_oof:.5f}**. Regla: **{'PASA' if PASA else 'NO PASA'}**, objetivo **{GATE_TARGET:.5f}**. {'Holdout evaluado una vez y CSV generado.' if PASA else 'Holdout y CSV omitidos explicitamente.'}

Se reutilizo el NPZ de v6, hash coincidente y grupos disjuntos, con los mismos cinco pliegues. V5 tenia fuga y un split diferente. Los resultados calibrados anteriores son optimistas. La seleccion de theta sobre OOF tambien introduce optimismo de seleccion. El score publico de Kaggle no es comparable directamente con holdout.

No se cambiaron archivos existentes ni se regenero la particion. No se hicieron commit, push ni envios a Kaggle.
"""
display(Markdown(summary_text))
assert INPUT_HASHES=={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in INPUT_HASHES}
assert PROTECTED_HASHES=={name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in PROTECTED_HASHES}
print('HASHES DE ARCHIVOS PROTEGIDOS: SIN CAMBIOS.')


,version,OOF_sin_calibrar,OOF_calibrado,holdout
0,v5 KN historica,0.435390,0.43847,0.44945
1,v6 KN agrupada,0.441060,0.44353,0.41847
2,v7 KN por cortes,0.423210,0.42468,NaN
3,v7 descuento-concentracion,0.440875,NaN,NaN


## Resumen final

Modelo: **interpolación descontada con parámetro de concentración, basada en Chen y Goodman (1999)**. Teh (2006) es solo inspiracion; no es Pitman-Yor exacto ni inferencia bayesiana.

Prueba de normalizacion: **PASA**, tolerancia **1e-9**, ambos theta, todos los historiales/ordenes en caracteres y palabras, con suma directa en el corpus sintetico y agregacion exacta de masa en los modelos reales. Error maximo real: **1.266e-13**.

Theta elegido: **0.5**; F1 OOF sin calibrar: **0.44088**. Regla: **NO PASA**, objetivo **0.44306**. Holdout y CSV omitidos explicitamente.

Se reutilizo el NPZ de v6, hash coincidente y grupos disjuntos, con los mismos cinco pliegues. V5 tenia fuga y un split diferente. Los resultados calibrados anteriores son optimistas. La seleccion de theta sobre OOF tambien introduce optimismo de seleccion. El score publico de Kaggle no es comparable directamente con holdout.

No se cambiaron archivos existentes ni se regenero la particion. No se hicieron commit, push ni envios a Kaggle.


HASHES DE ARCHIVOS PROTEGIDOS: SIN CAMBIOS.
